# Ideal-gas isotherms

## Prerequisites

Review the ideal-gas law $PV=nR_gT$ and the meaning of an isotherm.

## Objectives

1. Construct a unit-aware ideal-gas equation of state.
2. Evaluate pressure over explicit physical volumes at several temperatures.
3. Verify the represented equation numerically after unit conversion.
4. Plot and interpret a family of pressure–volume isotherms.

The model is illustrative and does not validate ideal-gas behavior for a particular material or pressure range.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.constants import SI
from projectkoios.physkit.thermal.thermo.eos.ideal_gas import (
    IdealGasEquationOfState,
)
from projectkoios.physkit.units import (
    MODEL_SYSTEM_UNIT_CONVERTER,
    PhysicalUnit,
    ScalarQuantity,
    VectorQuantity,
)

In [ ]:
amount_moles = 1.0
minimum_volume_liter = 0.1
maximum_volume_liter = 2.0
volume_sample_count = 500
temperature_values_kelvin = np.array(
    [200.0, 225.0, 250.0, 275.0, 300.0],
    dtype=np.float64,
)
volume_unit = PhysicalUnit(expression="liter")
temperature_unit = PhysicalUnit(expression="kelvin")
pressure_unit = PhysicalUnit(expression="bar")

model = IdealGasEquationOfState(
    amount=ScalarQuantity(
        magnitude=amount_moles,
        unit=PhysicalUnit(expression="mole"),
    )
)
volume_values_liter = np.linspace(
    minimum_volume_liter,
    maximum_volume_liter,
    volume_sample_count,
    dtype=np.float64,
)
volumes = VectorQuantity(magnitude=volume_values_liter, unit=volume_unit)
temperatures = VectorQuantity(
    magnitude=temperature_values_kelvin,
    unit=temperature_unit,
)
isotherms = model.evaluate_isotherms(
    volumes=volumes,
    temperatures=temperatures,
    pressure_unit=pressure_unit,
)

In [ ]:
pascal = PhysicalUnit(expression="pascal")
cubic_metre = PhysicalUnit(expression="meter ** 3")
equation_relative_tolerance = 3.0e-16
for isotherm in isotherms:
    pressures_pascal = MODEL_SYSTEM_UNIT_CONVERTER.convert_vector(
        quantity=isotherm.pressures,
        target=pascal,
    )
    volumes_cubic_metres = MODEL_SYSTEM_UNIT_CONVERTER.convert_vector(
        quantity=isotherm.volumes,
        target=cubic_metre,
    )
    represented_ratio = (
        pressures_pascal.magnitude
        * volumes_cubic_metres.magnitude
        / (
            model.amount_in_moles.magnitude
            * SI.R_g
            * isotherm.request.temperature_in_kelvin.magnitude
        )
    )
    assert np.allclose(
        represented_ratio,
        1.0,
        rtol=equation_relative_tolerance,
        atol=equation_relative_tolerance,
    )

In [ ]:
figure_size_inches = (8.0, 4.8)
figure, axis = plt.subplots(figsize=figure_size_inches)
for isotherm in isotherms:
    axis.plot(
        isotherm.volumes.magnitude,
        isotherm.pressures.magnitude,
        label=f"{isotherm.temperature.magnitude:.0f} K",
    )
axis.set(
    xlabel="volume (L)",
    ylabel="pressure (bar)",
    title="One-mole ideal-gas isotherms",
)
axis.grid(True)
axis.legend(title="temperature")
figure.tight_layout()
plt.show()

## Interpretation

At fixed temperature, pressure decreases inversely with volume. At fixed volume, the higher-temperature isotherm has higher pressure. The curves do not include intermolecular interactions, excluded volume, condensation, or phase coexistence.

## Exercises

1. Derive the slope $dP/dV$ along one ideal-gas isotherm.
2. Repeat the plot for a different amount of substance and predict the scaling.
3. Evaluate the same isotherms using cubic metres and pascals and verify that the physical curves are unchanged.
4. Explain why a general implicit equation-of-state solver is unnecessary for this explicit ideal-gas relation.
